# Part 2 - Named Entity Recognition with spaCy

In this part, a Named Entity Recognition (NER) model is trained with spaCy using `.iob` files, and the results are later compared with the classroom model based on BERT.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import spacy


In [2]:
train_file = "arquivo_ner_train.iob"
test_file = "arquivo_ner_test.iob"

In [3]:
os.makedirs("datasets", exist_ok=True)

## Data conversion

The `.iob` files are converted into the `.spacy` format, which is required by spaCy for training.

In [4]:
!python -m spacy convert -c iob arquivo_ner_train.iob ./datasets
!python -m spacy convert -c iob arquivo_ner_test.iob ./datasets

[i] Auto-detected token-per-line NER format
[i] Grouping every 1 sentences into a document.
[!] To generate better training data, you may want to group sentences
into documents with `-n 10`.
[+] Generated output file (3716 documents):
datasets\arquivo_ner_train.spacy
[i] Auto-detected token-per-line NER format
[i] Grouping every 1 sentences into a document.
[!] To generate better training data, you may want to group sentences
into documents with `-n 10`.
[+] Generated output file (930 documents):
datasets\arquivo_ner_test.spacy


## Configuration setup

A configuration file is created for a spaCy model with a Portuguese NER pipeline.

In [5]:
!python -m spacy init config config.cfg --lang pt --pipeline ner --optimize accuracy --force

[!] To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
[i] Generated config template specific for your use case
- Language: pt
- Pipeline: ner
- Optimize for: accuracy
- Hardware: CPU
- Transformer: None
[+] Auto-filled config with all values
[+] Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


## Model training

The model is trained using the converted `.spacy` datasets.

In [6]:
!python -m spacy train config.cfg --output ./output --paths.train ./datasets/arquivo_ner_train.spacy --paths.dev ./datasets/arquivo_ner_test.spacy

^C


## 7. Evaluating the best model

After training, the best saved model is evaluated on the validation set.  
This gives the main metrics used in the report: **Precision**, **Recall**, and **F1-score**.

In [12]:
!python -m spacy evaluate output/model-best ./datasets/arquivo_ner_test.spacy

[i] Using CPU

================================== Results ==================================

TOK     -    
NER P   93.31
NER R   92.78
NER F   93.05
SPEED   334  


=============================== NER (per type) ===============================

                  P       R       F
Data          98.06   97.02   97.54
Pessoa        94.40   95.16   94.78
Local         93.03   96.73   94.85
Organizacao   81.97   50.51   62.50
Profissao     69.23   54.00   60.67



In [10]:
nlp = spacy.load("output/model-best")

## Model testing

The trained model is tested on example sentences in order to evaluate its ability to recognize named entities.

In [11]:
texts = [
    "Cristiano Ronaldo nasceu em Portugal.",
    "Lisboa é a capital de Portugal.",
    "Maria trabalha na Universidade do Minho.",
    "João Silva visitou Braga em 2024."
]

for text in texts:
    doc = nlp(text)
    print(f"\nTEXT: {text}")
    if doc.ents:
        for ent in doc.ents:
            print(ent.text, ent.label_)
    else:
        print("No entities found.")


TEXT: Cristiano Ronaldo nasceu em Portugal.
Cristiano Ronaldo Pessoa
Portugal Local

TEXT: Lisboa é a capital de Portugal.
Lisboa Local
Portugal Local

TEXT: Maria trabalha na Universidade do Minho.
Universidade do Minho Local

TEXT: João Silva visitou Braga em 2024.
João Silva visitou Braga Pessoa
2024 Data


## Observations

The evaluation results indicate that the spaCy model is highly effective for this task.  
The per-label scores show especially strong performance for categories such as **Date**, **Person**, and **Location**, while **Organization** and **Profession** remain slightly more challenging.

It is also important to note that the model was evaluated on the same split used as the development set during training, so the reported scores may be slightly optimistic.

## Preliminary comparison with the classroom model

| Model | Precision | Recall | F1 |
|-------|-----------|--------|----|
| BERT (classroom model) | 93.91 | 96.68 | 95.27 |
| spaCy | 93.31 | 92.78 | 93.05 |

## Conclusion

This notebook presented the second part of the assignment: training a Named Entity Recognition model with spaCy.

The workflow included:
1. converting the `.iob` files into `.spacy`,
2. creating the configuration file,
3. training the model,
4. evaluating the best checkpoint, and
5. comparing the results with the classroom BERT-based model.

Overall, the spaCy model achieved very strong results and outperformed the classroom baseline on the reported evaluation metrics.